# 1. LLM Gateway

*Estimated time to run notebook: about 5 min*

**Goal:** Establish the foundation for our agent by connecting to the **DataRobot LLM Gateway**.

**Key Concept:**
Instead of managing API keys for every provider (Azure, AWS Bedrock, Google Vertex), DataRobot provides a single, unified endpoint. In this notebook, we verify access to nearly 100 different LLMs and select a base model (`azure/gpt-5-1-2025-11-13` or similar) to power our agent's reasoning capabilities.

In [6]:
# Install required packages
!uv pip install -q -r requirements.txt

### Connect to DataRobot LLM Gateway and inspect number of available LLMs

In [7]:
import datarobot as dr
from pprint import pprint

# 1. Initialize client
# This uses your local/notebook DataRobot credentials.
dr_client = dr.Client()

# 2. Query the LLM Gateway catalog
response = dr_client.get(url="genai/llmgw/catalog/")

# 3. Extract supported model IDs
# The catalog returns a list of model metadata; we keep just the string identifiers.
data = response.json()["data"]
supported_llms = [llm_model["model"] for llm_model in data]

# 4. Inspect results
print("Number of LLMs supported by LLM Gateway:", len(supported_llms))
pprint(supported_llms)

### Select a model to use the DataRobot LLM Gateway and ask a question

In [8]:
import os
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# 1. Load configuration
# .env is optional; defaults are provided below.
load_dotenv()

# 2. Configure the model (via DataRobot LLM Gateway)
MODEL_NAME = 'azure/gpt-5-4-2026-03-05'
#MODEL_NAME = os.getenv("MODEL_NAME")
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token,
        base_url=dr_client.endpoint + "/genai/llmgw",
    ),
)

# 3. Define the agent
agent = Agent(model=model)

# 4. Execution (sanity-check)
response = await agent.run("What is the capital of France?")
pprint(response.output)

⚠️ Note on Kernel Limits: DataRobot Codespaces have a limit of 5 active notebook kernels at a time. To ensure a smooth transition to the next exercise, please remember to shut down this kernel (by closing the notebook tab) once you are finished. This prevents any 'limit reached' errors when opening subsequent notebooks!